In [1]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
)


In [2]:
# ---------------------------------------------------------------------------
# 1) CONFIGURACIÓN — ajusta rutas si es necesario
# ---------------------------------------------------------------------------
DATASET_ROOT = Path(r"D:\Publicaciones\SIMEE\dataset_final")
OUTPUT_DIR = Path(r"D:\Publicaciones\SIMEE\inceptionv3_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (299, 299)
BATCH_SIZE = 8
CLASS_NAMES = ["calcification", "mass", "normal"]  # orden alfabético = orden de flow_from_directory

# --- Valores NO especificados explícitamente en el paper original ---
# Documentados aquí para poder citarlos en la sección de Métodos revisada.
DROPOUT_RATE = 0.5          # asumido; el paper no reporta la tasa exacta
STAGE1_LR = 1e-4            # asumido; el paper no reporta el LR inicial de InceptionV3
STAGE2_LR = 1e-5            # LR reducido para fine-tuning (asumido, 10x menor)
EARLY_STOPPING_PATIENCE = 5  # asumido; el paper no reporta la paciencia usada
UNFREEZE_FROM_LAYER = "mixed7"  # a partir de qué bloque se descongela en la Etapa 2

STAGE1_MAX_EPOCHS = 30
STAGE2_MAX_EPOCHS = 20


In [3]:
# ---------------------------------------------------------------------------
# SEMILLA ÚNICA — se aplica a TODAS las fuentes de aleatoriedad desde aquí.
# No declares otra semilla en ningún otro punto del script; cambia solo esta.
# ---------------------------------------------------------------------------
SEED = 42


def set_global_seed(seed: int):
    import os
    import random
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        # Fuerza operaciones deterministas cuando el backend lo soporta (TF >= 2.9).
        # En algunas capas/GPUs puede no haber kernel determinista disponible;
        # si falla, seguimos igual pero con menor garantía de reproducibilidad exacta.
        tf.config.experimental.enable_op_determinism()
    except Exception as e:
        print(f"[aviso] No se pudo forzar determinismo total de operaciones: {e}")


set_global_seed(SEED)


In [4]:
# ---------------------------------------------------------------------------
# DETECCIÓN DE HARDWARE — ajusta la configuración automáticamente a tu equipo,
# sin tocar nada a mano. No cambia el BATCH_SIZE reportado en el paper (8),
# solo cómo TensorFlow usa la GPU disponible (si la hay).
# ---------------------------------------------------------------------------
gpus = tf.config.list_physical_devices("GPU")
USE_MIXED_PRECISION = False

if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(f"[aviso] No se pudo activar memory growth en {gpu}: {e}")
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        USE_MIXED_PRECISION = True
    except Exception as e:
        print(f"[aviso] No se pudo activar mixed precision: {e}")
    print(f"[hardware] GPU(s) detectada(s): {[g.name for g in gpus]} | mixed_precision={USE_MIXED_PRECISION}")
else:
    print("[hardware] No se detectó GPU — entrenando en CPU (será más lento).")

[hardware] No se detectó GPU — entrenando en CPU (será más lento).


In [5]:
# ---------------------------------------------------------------------------
# 2) Generadores de datos (misma augmentación reportada en el paper, Sección II.C)
# ---------------------------------------------------------------------------
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255)  # sin augmentación en val/test

train_gen = train_datagen.flow_from_directory(
    DATASET_ROOT / "train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=True,
    seed=SEED,
)
val_gen = eval_datagen.flow_from_directory(
    DATASET_ROOT / "val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False,
)
test_gen = eval_datagen.flow_from_directory(
    DATASET_ROOT / "test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False,
)

print(f"Train: {train_gen.samples} imágenes | Val: {val_gen.samples} | Test: {test_gen.samples}")
print(f"Mapeo de clases: {train_gen.class_indices}")


Found 3719 images belonging to 3 classes.
Found 831 images belonging to 3 classes.
Found 961 images belonging to 3 classes.
Train: 3719 imágenes | Val: 831 | Test: 961
Mapeo de clases: {'calcification': 0, 'mass': 1, 'normal': 2}


In [6]:
# ---------------------------------------------------------------------------
# 3) Arquitectura (replica exacta de la Tabla II del paper)
# ---------------------------------------------------------------------------
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # Etapa 1: base congelada

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256)(x)
x = BatchNormalization()(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)
predictions = Dense(len(CLASS_NAMES), activation="softmax", dtype="float32")(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 22,361,635 (85.30 MB)

 Trainable params: 558,339 (2.13 MB)

 Non-trainable params: 21,803,296 (83.17 MB)

In [7]:
# ---------------------------------------------------------------------------
# 4) ETAPA 1 — entrenar solo las capas nuevas
# ---------------------------------------------------------------------------
model.compile(optimizer=Adam(learning_rate=STAGE1_LR), loss="categorical_crossentropy", metrics=["accuracy"])

stage1_ckpt = OUTPUT_DIR / "stage1_best.keras"
callbacks_stage1 = [
    EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ModelCheckpoint(str(stage1_ckpt), monitor="val_loss", save_best_only=True, verbose=1),
    CSVLogger(str(OUTPUT_DIR / "stage1_history.csv")),
]

print("\n=== ETAPA 1: entrenando capas superiores (base congelada) ===")
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=STAGE1_MAX_EPOCHS,
    callbacks=callbacks_stage1,
)
stage1_stopped_epoch = len(history1.history["loss"])
print(f"Etapa 1 detenida en la época {stage1_stopped_epoch} de {STAGE1_MAX_EPOCHS} "
      f"(patience={EARLY_STOPPING_PATIENCE}, monitor=val_loss)")



=== ETAPA 1: entrenando capas superiores (base congelada) ===
Epoch 1/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 764ms/step - accuracy: 0.4551 - loss: 1.6073
Epoch 1: val_loss improved from None to 0.58851, saving model to D:\Publicaciones\SIMEE\inceptionv3_run\stage1_best.keras
465/465 ━━━━━━━━━━━━━━━━━━━━ 434s 920ms/step - accuracy: 0.5190 - loss: 1.3473 - val_accuracy: 0.6811 - val_loss: 0.5885 - learning_rate: 1.0000e-04
Epoch 2/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 752ms/step - accuracy: 0.5968 - loss: 1.0195
Epoch 2: val_loss improved from 0.58851 to 0.51434, saving model to D:\Publicaciones\SIMEE\inceptionv3_run\stage1_best.keras
465/465 ━━━━━━━━━━━━━━━━━━━━ 418s 899ms/step - accuracy: 0.6168 - loss: 1.0006 - val_accuracy: 0.7244 - val_loss: 0.5143 - learning_rate: 1.0000e-04
Epoch 3/30
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 779ms/step - accuracy: 0.6328 - loss: 0.9679
Epoch 3: val_loss improved from 0.51434 to 0.51140, saving model to D:\Publicaciones\SIMEE\inceptionv3_run\stage1_best.keras
465

In [8]:
# ---------------------------------------------------------------------------
# 5) ETAPA 2 — fine-tuning: descongelar capas profundas
# ---------------------------------------------------------------------------
unfreeze = False
for layer in base_model.layers:
    if layer.name == UNFREEZE_FROM_LAYER:
        unfreeze = True
    layer.trainable = unfreeze

model.compile(optimizer=Adam(learning_rate=STAGE2_LR), loss="categorical_crossentropy", metrics=["accuracy"])

stage2_ckpt = OUTPUT_DIR / "stage2_best.keras"
callbacks_stage2 = [
    EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    ModelCheckpoint(str(stage2_ckpt), monitor="val_loss", save_best_only=True, verbose=1),
    CSVLogger(str(OUTPUT_DIR / "stage2_history.csv")),
]

print(f"\n=== ETAPA 2: fine-tuning desde la capa '{UNFREEZE_FROM_LAYER}' ===")
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=STAGE2_MAX_EPOCHS,
    callbacks=callbacks_stage2,
)
stage2_stopped_epoch = len(history2.history["loss"])
print(f"Etapa 2 detenida en la época {stage2_stopped_epoch} de {STAGE2_MAX_EPOCHS} "
      f"(patience={EARLY_STOPPING_PATIENCE}, monitor=val_loss)")



=== ETAPA 2: fine-tuning desde la capa 'mixed7' ===
Epoch 1/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - accuracy: 0.6657 - loss: 0.7117
Epoch 1: val_loss improved from None to 0.44237, saving model to D:\Publicaciones\SIMEE\inceptionv3_run\stage2_best.keras
465/465 ━━━━━━━━━━━━━━━━━━━━ 531s 1s/step - accuracy: 0.6819 - loss: 0.6699 - val_accuracy: 0.7449 - val_loss: 0.4424 - learning_rate: 1.0000e-05
Epoch 2/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 954ms/step - accuracy: 0.7081 - loss: 0.6022
Epoch 2: val_loss improved from 0.44237 to 0.42885, saving model to D:\Publicaciones\SIMEE\inceptionv3_run\stage2_best.keras
465/465 ━━━━━━━━━━━━━━━━━━━━ 513s 1s/step - accuracy: 0.7023 - loss: 0.6071 - val_accuracy: 0.7509 - val_loss: 0.4289 - learning_rate: 1.0000e-05
Epoch 3/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 0s 960ms/step - accuracy: 0.7081 - loss: 0.5657
Epoch 3: val_loss improved from 0.42885 to 0.41077, saving model to D:\Publicaciones\SIMEE\inceptionv3_run\stage2_best.keras
465/465 ━━━━━━━━━━━

In [9]:
# ---------------------------------------------------------------------------
# 6) Evaluación en TEST + guardar TODAS las predicciones (para CI/FROC/calibración después)
# ---------------------------------------------------------------------------
test_gen.reset()
y_true = test_gen.classes
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

filenames = test_gen.filenames
pred_df = pd.DataFrame({
    "filename": filenames,
    "true_class": [CLASS_NAMES[i] for i in y_true],
    "pred_class": [CLASS_NAMES[i] for i in y_pred],
})
for i, cname in enumerate(CLASS_NAMES):
    pred_df[f"prob_{cname}"] = y_pred_probs[:, i]

pred_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
print(f"\nPredicciones completas del test guardadas en {OUTPUT_DIR / 'test_predictions.csv'}")
print("(esto es lo que se usará después para calcular CI, calibración y FROC sin reentrenar)")

test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")


121/121 ━━━━━━━━━━━━━━━━━━━━ 91s 742ms/step

Predicciones completas del test guardadas en D:\Publicaciones\SIMEE\inceptionv3_run\test_predictions.csv
(esto es lo que se usará después para calcular CI, calibración y FROC sin reentrenar)

Test accuracy: 0.7399 | Test loss: 0.4559


In [10]:
# ---------------------------------------------------------------------------
# 7) Guardar metadatos de la corrida (para citar en la sección de Métodos revisada)
# ---------------------------------------------------------------------------
run_metadata = {
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "gpu_detected": [g.name for g in gpus] if gpus else [],
    "mixed_precision_enabled": USE_MIXED_PRECISION,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "dropout_rate_assumed": DROPOUT_RATE,
    "stage1_lr_assumed": STAGE1_LR,
    "stage2_lr_assumed": STAGE2_LR,
    "early_stopping_patience_assumed": EARLY_STOPPING_PATIENCE,
    "unfreeze_from_layer": UNFREEZE_FROM_LAYER,
    "stage1_stopped_epoch": stage1_stopped_epoch,
    "stage1_max_epochs": STAGE1_MAX_EPOCHS,
    "stage2_stopped_epoch": stage2_stopped_epoch,
    "stage2_max_epochs": STAGE2_MAX_EPOCHS,
    "train_samples": train_gen.samples,
    "val_samples": val_gen.samples,
    "test_samples": test_gen.samples,
    "class_indices": train_gen.class_indices,
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
}
with open(OUTPUT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2)

model.save(OUTPUT_DIR / "inceptionv3_final.keras")
print(f"\nModelo y metadatos guardados en {OUTPUT_DIR}")



Modelo y metadatos guardados en D:\Publicaciones\SIMEE\inceptionv3_run
